# Import ICDE demo CSV sample package

Uses **w4h-sample-icde-demo-csv** + your personal API key.

1. Admin creates the dataset in the app first
2. `cp .env.example .env` and set `W4H_API_KEY`
3. Run the cells: **pick dataset → review gps_track mapping → import**

The first code cell lists datasets via `GET /datasets`. In-app Run does not use this notebook — the Admin picks the dataset in the dialog and the runner passes that id into `run.sh`.

Unlike the Fitbit sample, this package includes a `gps_track` signal — its `manifest.yaml` entry maps `geo: {lat: latitude, lon: longitude}` rather than `geo: null`.

In [ ]:
import ipywidgets as widgets

from w4h_sample_icde_demo_csv_importer.client import W4HClient
from w4h_sample_icde_demo_csv_importer.config import sample_package_path
from w4h_sample_icde_demo_csv_importer.csv_import import import_all
from w4h_sample_icde_demo_csv_importer.manifest import load_manifest, signal_by_slug

package_root = sample_package_path()
manifest = load_manifest(package_root)
client = W4HClient()
datasets = client.list_datasets()
assert datasets, "No datasets visible for this API key — create one in the Admin UI first."
ds_picker = widgets.Dropdown(
    options=[(f"{d.get('name') or d.get('id')}  ({d.get('id')})", d.get("id")) for d in datasets],
    description="Dataset:",
)
print("package:", package_root)
print(f"{len(datasets)} dataset(s) this key can see. Pick one, then continue.")
ds_picker

In [ ]:
# Preview the geo signal's mapping before importing anything.
gps = signal_by_slug(manifest, "gps_track")
gps["geomts_mapping"]

In [ ]:
ds_id = ds_picker.value
assert ds_id, "Pick a dataset in the first code cell."
results = import_all(client, ds_id, manifest, package_root, mode="replace")
results